# Lab 5: 规划与协调层 — Planning & Coordination

对应 Harness 六层架构：第 ❺ 层

## 学习目标

- 理解 **Team Lead 协调模式** — 主 agent 分派子任务给子 agent
- 实现 `AgentTool`：让 agent 可以"分身"，生成拥有独立 context 的子 agent
- 实现任务管理系统：创建、跟踪、完成子任务
- 理解子 agent **独立 context** 为什么重要

## 痛点回顾

Lab 4 的 agent 有了安全拦截层。但面对复杂的多步骤任务时：
- 只能串行处理，一个接一个
- 所有子任务的上下文挤在同一个 context window 中
- 无法并行或分而治之

**本 Lab 的目标：让 agent 学会"分身术"和"团队协作"。**

## 对应 Claude Code 源码

- `tools/AgentTool/` — 子 agent 生成工具
- `tools/TeamCreateTool/` — 团队创建
- `tools/TaskCreateTool/` — 任务创建
- `tools/SendMessageTool/` — Agent 间通信
- `coordinator/` — 多 agent 协调模式

Claude Code 的多 agent 架构：

```
Team Lead (主 agent)
  ├── TaskCreate    → 拆分子任务
  ├── AgentTool     → 生成子 agent（独立 context）
  ├── SendMessage   → agent 间通信
  └── TaskUpdate    → 跟踪任务状态

Sub-Agent (子 agent)
  ├── 拥有基础工具（read/write/bash）
  ├── 独立的 messages 历史
  ├── 有限的轮数（防止失控）
  └── 完成后返回结果给主 agent
```


---
## 第一步：环境准备 + 回顾 Lab 2-4 代码

In [2]:
# === 环境准备 + Lab 2-4 核心代码回顾 ===
from anthropic import AnthropicBedrock

client = AnthropicBedrock(aws_region="us-west-2")
MODEL = "global.anthropic.claude-sonnet-4-6"

# --- 三个核心工具（来自 Lab 2)---
from utils.tools import WriteFileTool,ReadFileTool,BashTool,Tool
# --- 权限系统（预览，完整版见 Lab 3）---
from utils.permissions import SmartPermissionChecker
DANGEROUS_PATTERNS = ["rm ", "sudo ", "chmod ", "chown ", "mkfs", "dd if="]
SAFE_PATTERNS = ["ls", "cat ", "head ", "tail ", "echo ", "pwd", "python ", "python3 ",
                 "git status", "git log", "wc ", "grep ", "find "]

BASE_TOOLS = [ReadFileTool(), WriteFileTool(), BashTool()]
checker = SmartPermissionChecker()
print(f"✓ Lab 2-4 代码回顾完成，{len(BASE_TOOLS)} 个基础工具就绪")

✓ Lab 2-4 代码回顾完成，3 个基础工具就绪


---
## 第二步：构建子 Agent 运行器

在实现 `AgentTool` 之前，我们先抽取一个通用的 agent 运行函数。

这个函数将被主 agent 和子 agent 共用——子 agent 就是一个运行在独立 context 中的 agent loop。

In [3]:
def run_agent(system_prompt, tools_list, user_messages,
              max_turns=20, permission_checker=None, context_manager=None):
    """通用 Agent 运行器（消息列表驱动）。主 agent 和子 agent 都使用此函数。"""
    def _serialize_content(content_blocks):
        """将 SDK ContentBlock 序列化为纯 dict 列表"""
        result = []
        for blk in content_blocks:
            if hasattr(blk, 'type'):
                if blk.type == 'text':
                    result.append({'type': 'text', 'text': blk.text})
                elif blk.type == 'tool_use':
                    result.append({'type': 'tool_use', 'id': blk.id, 'name': blk.name, 'input': blk.input})
                else:
                    result.append({'type': blk.type})
            elif isinstance(blk, dict):
                result.append(blk)
        return result

    _tool_map = {t.name: t for t in tools_list}
    _tools_schema = [t.to_api_schema() for t in tools_list]
    messages = []
    last_text = ""
    msg_index = 0

    for turn in range(1, max_turns + 1):
        if msg_index >= len(user_messages):
            break
        user_input = user_messages[msg_index]
        msg_index += 1

        messages.append({"role": "user", "content": user_input})

        # 上下文压缩检查
        if context_manager and context_manager.should_compact(messages):
            messages = context_manager.compact(messages)
            print("  [Compaction] 上下文已压缩")

        while True:
            response = client.messages.create(
                model=MODEL, max_tokens=4096, system=system_prompt,
                tools=_tools_schema, messages=messages,
            )
            messages.append({"role": "assistant", "content": _serialize_content(response.content)})

            tool_results = []
            for blk in response.content:
                if blk.type == "text":
                    last_text = blk.text
                elif blk.type == "tool_use":
                    tool = _tool_map.get(blk.name)
                    if tool is None:
                        result, is_error = f"未知工具 {blk.name}", True
                    elif not tool.validate(blk.input):
                        result, is_error = "参数校验失败", True
                    else:
                        perm = permission_checker.check(blk.name, blk.input) if permission_checker else "allow"
                        if perm == "deny":
                            result, is_error = "权限拒绝", True
                        elif perm == "ask":
                            result, is_error = tool.execute(blk.input), False
                        else:
                            result, is_error = tool.execute(blk.input), False

                    tool_results.append({
                        "type": "tool_result", "tool_use_id": blk.id,
                        "content": result, **(dict(is_error=True) if is_error else {}),
                    })

            if tool_results:
                messages.append({"role": "user", "content": tool_results})
            if response.stop_reason != "tool_use":
                break

    return last_text

print("run_agent 就绪（支持 permission_checker + context_manager）")


run_agent 就绪（支持 permission_checker + context_manager）


---
## 第三步：实现 AgentTool — 子 Agent 生成器

这是多 agent 系统的核心：**让 agent 可以"分身"**。

当主 agent 调用 `dispatch_agent` 工具时：
1. 创建一个新的 `run_agent()` 实例
2. 给它传入任务描述作为 initial_message
3. 它拥有**独立的 messages 列表**（独立 context）
4. 它只能使用基础工具（**不能再分身**，防止递归）
5. 有最大轮数限制（防止失控）
6. 执行完毕后，将结果返回给主 agent

这对应 Claude Code 中的 `AgentTool`。

In [4]:
# AgentTool: 子 Agent 生成器（对应 Claude Code 的 tools/AgentTool/）

class AgentTool(Tool):
    """生成子 agent 来处理子任务。子 agent 拥有独立的 context。"""

    def __init__(self, base_tools):
        self._base_tools = base_tools

    @property
    def name(self): return "dispatch_agent"

    @property
    def description(self):
        return "派遣一个子 agent 来独立完成一个具体的子任务。子 agent 拥有独立的上下文。请提供清晰具体的任务描述。"

    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "task": {"type": "string", "description": "要派发给子 agent 的具体任务描述"},
            },
            "required": ["task"],
        }

    def execute(self, tool_input):
        task = tool_input["task"]
        print(f"\n  子Agent启动: {task[:80]}...")

        sub_prompt = f"你是一个子 agent。请直接完成任务并给出简洁的结果报告。\n任务: {task}"

        # 子 agent 用消息列表驱动（单条任务消息）
        result = run_agent(
            system_prompt=sub_prompt,
            tools_list=self._base_tools,
            user_messages=[task],
            max_turns=5,
        )

        print(f"  子Agent完成: {result[:100]}...")
        return result

print("AgentTool 就绪")


AgentTool 就绪


---
## 第四步：实现任务管理系统

Claude Code 中，Team Lead 使用任务系统来：
1. **拆分** — 把复杂任务拆成可独立执行的子任务
2. **跟踪** — 查看哪些任务完成了，哪些还在进行
3. **协调** — 确保子任务之间的依赖关系得到满足

我们实现一个简化版的 `TaskCreate` 和 `TaskList`。

In [5]:
# 任务管理器（对应 Claude Code 的 TaskCreateTool / TaskListTool）

class TaskManager:
    """简单的任务管理系统。"""
    
    def __init__(self):
        self._tasks: dict[str, dict] = {}
        self._counter = 0
    
    def create(self, subject: str, description: str = "") -> str:
        self._counter += 1
        task_id = str(self._counter)
        self._tasks[task_id] = {
            "subject": subject,
            "description": description,
            "status": "pending",
        }
        return task_id
    
    def update(self, task_id: str, status: str):
        if task_id in self._tasks:
            self._tasks[task_id]["status"] = status
    
    def list_all(self) -> list[dict]:
        return [{"id": k, **v} for k, v in self._tasks.items()]

# 全局任务管理器实例
task_mgr = TaskManager()

# 任务创建工具
class TaskCreateTool(Tool):
    @property
    def name(self): return "create_task"
    @property
    def description(self): return "创建一个子任务来跟踪工作进度"
    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "subject": {"type": "string", "description": "任务标题"},
                "description": {"type": "string", "description": "任务详情"},
            },
            "required": ["subject"],
        }
    def execute(self, tool_input):
        tid = task_mgr.create(tool_input["subject"], tool_input.get("description", ""))
        return f"任务 #{tid} 已创建: {tool_input['subject']}"

# 任务列表工具
class TaskListTool(Tool):
    @property
    def name(self): return "list_tasks"
    @property
    def description(self): return "列出所有任务及其状态"
    @property
    def input_schema(self):
        return {"type": "object", "properties": {}}
    def execute(self, tool_input):
        tasks = task_mgr.list_all()
        if not tasks:
            return "当前没有任务。"
        lines = []
        for t in tasks:
            icon = "✅" if t["status"] == "completed" else "⏳" if t["status"] == "in_progress" else "📋"
            lines.append(f"{icon} #{t['id']} [{t['status']}] {t['subject']}")
        return "\n".join(lines)

# 任务更新工具
class TaskUpdateTool(Tool):
    @property
    def name(self): return "update_task"
    @property
    def description(self): return "更新任务状态（pending → in_progress → completed）"
    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "task_id": {"type": "string", "description": "任务 ID"},
                "status": {"type": "string", "enum": ["pending", "in_progress", "completed"]},
            },
            "required": ["task_id", "status"],
        }
    def execute(self, tool_input):
        task_mgr.update(tool_input["task_id"], tool_input["status"])
        return f"任务 #{tool_input['task_id']} 状态已更新为 {tool_input['status']}"

print("✓ 任务管理工具就绪: create_task, list_tasks, update_task")

✓ 任务管理工具就绪: create_task, list_tasks, update_task


---
## 第四步（续）：实现 SendMessageTool — Agent 间通信

Claude Code 中，Team Lead 和子 Agent 之间通过 `SendMessageTool` 通信。
这是多 Agent 协作的关键基础设施：

- Team Lead 可以给子 Agent 发指令
- 子 Agent 可以向 Team Lead 汇报进度
- Agent 间通过共享的 **Mailbox（消息邮箱）** 交换消息

Claude Code 使用文件系统（`TeamFile` + `Mailbox`）实现 Agent 间通信。
我们用一个简化的内存 Mailbox 来演示。


In [6]:
# SendMessageTool: Agent 间通信（对应 Claude Code 的 SendMessageTool）

from datetime import datetime

class Mailbox:
    """Agent 间的消息邮箱。类似 Claude Code 中基于文件的 Mailbox 机制。"""
    def __init__(self):
        self._messages: list[dict] = []

    def send(self, sender: str, receiver: str, content: str):
        self._messages.append({
            'sender': sender,
            'receiver': receiver,
            'content': content,
            'time': datetime.now().isoformat(),
        })

    def receive(self, receiver: str) -> list[dict]:
        """获取发给指定 receiver 的所有未读消息"""
        msgs = [m for m in self._messages if m['receiver'] == receiver]
        # 取出后清除（模拟已读）
        self._messages = [m for m in self._messages if m['receiver'] != receiver]
        return msgs

    def all_messages(self) -> list[dict]:
        """查看全部消息历史（调试用）"""
        return list(self._messages)


# 全局邮箱实例
mailbox = Mailbox()


class SendMessageTool(Tool):
    """发送消息给其他 Agent。对应 Claude Code 的 SendMessageTool。"""

    @property
    def name(self): return "send_message"
    @property
    def description(self):
        return "发送消息给其他 agent。用于 Team Lead 与子 agent 之间的通信。"
    @property
    def input_schema(self):
        return {
            "type": "object",
            "properties": {
                "to": {"type": "string", "description": "接收者名称（如 team-lead, sub-agent-1）"},
                "message": {"type": "string", "description": "消息内容"},
            },
            "required": ["to", "message"],
        }

    def execute(self, tool_input):
        receiver = tool_input['to']
        content = tool_input['message']
        mailbox.send('agent', receiver, content)
        print(f'    [SendMessage] -> {receiver}: {content[:100]}')
        return f'消息已发送给 {receiver}'


# === 演示：Agent 间通信 ===
print('=== SendMessage 演示 ===')

# 模拟 Team Lead 给子 Agent 发任务
mailbox.send('team-lead', 'sub-agent-1', '请分析 lab1 的代码结构')
mailbox.send('team-lead', 'sub-agent-2', '请分析 lab2 的代码结构')

# 模拟子 Agent 回复
mailbox.send('sub-agent-1', 'team-lead', 'lab1 有 17 个 cell，主要实现提示引导层')

# Team Lead 查收消息
msgs = mailbox.receive('team-lead')
print(f'\nTeam Lead 收到 {len(msgs)} 条消息:')
for m in msgs:
    print(f'  来自 {m["sender"]}: {m["content"]}')

# 重置邮箱
mailbox = Mailbox()
print('\n✓ SendMessageTool + Mailbox 就绪')


=== SendMessage 演示 ===

Team Lead 收到 1 条消息:
  来自 sub-agent-1: lab1 有 17 个 cell，主要实现提示引导层

✓ SendMessageTool + Mailbox 就绪


---
## 第五步：组装Mini Harness

现在把所有组件拼在一起！

**关键设计：**
- 主 agent（Team Lead）拥有全部工具：基础工具 + AgentTool + 任务工具
- 子 agent 只拥有基础工具（不能再分身，防止递归）
- 主 agent 可以自主决定是否拆分任务、是否派遣子 agent

In [7]:
# 组装完整的 Mini Harness

# 基础工具（子 agent 可用）
base_tools = [ReadFileTool(), WriteFileTool(), BashTool()]

# 主 agent 的完整工具集
all_tools = base_tools + [
    AgentTool(base_tools=base_tools),  # 子 agent 只能用基础工具
    TaskCreateTool(),
    TaskListTool(),
    TaskUpdateTool(),
    SendMessageTool(),
]

print("=== Mini Harness 架构 ===")
print(f"\n主 Agent (Team Lead) 工具:")
for t in all_tools:
    print(f"  🔧 {t.name}")
print(f"\n子 Agent 工具:")
for t in base_tools:
    print(f"  🔧 {t.name}")
print(f"\n注意: 子 agent 没有 dispatch_agent 工具（防递归）")


=== Mini Harness 架构 ===

主 Agent (Team Lead) 工具:
  🔧 read_file
  🔧 write_file
  🔧 bash
  🔧 dispatch_agent
  🔧 create_task
  🔧 list_tasks
  🔧 update_task
  🔧 send_message

子 Agent 工具:
  🔧 read_file
  🔧 write_file
  🔧 bash

注意: 子 agent 没有 dispatch_agent 工具（防递归）


---
## 第六步：验证运行

试试这个复杂任务，观察主 agent 如何拆分和协调：

> "分析当前目录下的所有 .ipynb 文件，报告每个文件的 cell 数量和主要内容摘要"

你应该能观察到：
1. 主 agent 创建任务计划
2. 主 agent 为每个文件派遣子 agent
3. 子 agent 独立分析各自的文件
4. 主 agent 汇总所有结果

In [ ]:
# 运行完整的 Mini Harness

TEAM_LEAD_PROMPT = """你是一个 Team Lead Agent（团队负责人）。

你的工作方式：
1. 收到复杂任务时，先用 create_task 拆分为子任务
2. 用 dispatch_agent 派遣子 agent 来处理各个子任务
3. 用 update_task 跟踪任务进度
4. 最后汇总所有结果给用户

对于简单任务，你可以直接用基础工具自己完成。
用中文回答。"""

# 重置任务管理器
task_mgr = TaskManager()

print("=" * 60)
print("Lab 5: Mini Harness — Team Lead Agent")
print("=" * 60)

run_agent(
    system_prompt=TEAM_LEAD_PROMPT,
    tools_list=all_tools,
    user_messages=[
        "分析当前目录下的所有 .ipynb 文件，报告每个文件的 cell 数量和主要内容摘要",
    ],
    permission_checker=checker,
)


Lab 5: Mini Harness — Team Lead Agent


---
## 完整集成示例：前五层 Harness 联动

集成 Lab 1-5 的全部能力（Lab 6 的状态持久层将在下一个 Lab 加入）：
- ❶ 提示与引导层（CLAUDE.md + Hooks）
- ❷ 工具与执行层（真实执行）
- ❸ 验证与安全层（权限检查 + Bash 分类器）
- ❹ 上下文与内存层（Compaction）
- ❺ 规划与协调层（AgentTool + TaskManager）


In [ ]:
# === 集成示例：❶提示 + ❷工具 + ❸安全 + ❹上下文 + ❺协调 ===
# 从 utils/ 导入前面所有 Lab 的实现

import os
from utils.project_memory import ProjectMemory
from utils.hooks import HookRunner
from utils.permissions import SmartPermissionChecker
from utils.context import ContextManager

# ❶ 项目记忆注入
pm = ProjectMemory()
full_prompt = pm.build_system_prompt(TEAM_LEAD_PROMPT, os.getcwd())
print(f'System prompt: {len(full_prompt)} 字符（含项目记忆）')

# ❶ Hooks: 工具调用统计
call_stats = {}
def stats_hook(name, inp):
    call_stats[name] = call_stats.get(name, 0) + 1
    return inp
hooks = HookRunner()
hooks.register_pre_tool(stats_hook)

# ❸ 权限检查
perm_checker = SmartPermissionChecker()

# ❹ 上下文管理
ctx = ContextManager(max_tokens=50000)

# ❺ 重置任务管理器
task_mgr = TaskManager()

# 注意：run_agent 目前不接受 context_manager 参数
# 这里演示的是组件的组合能力，完整集成将在 Lab 6 中实现
print('=' * 60)
print('集成示例：❶提示 + ❷工具 + ❸安全 + ❹上下文 + ❺协调')
print('=' * 60)

run_agent(
    system_prompt=full_prompt,
    tools_list=all_tools,
    user_messages=[
        '分析 CLAUDE.md 的内容，并用 bash 统计当前目录有多少个文件。把这些作为子任务来处理。',
    ],
    permission_checker=perm_checker,
)

# 统计
print(f'\n--- 工具调用统计 ---')
for name, count in sorted(call_stats.items()):
    print(f'  {name}: {count} 次')
print(f'\n--- 任务列表 ---')
for t in task_mgr.list_all():
    print(f'  {t["status"]} #{t["id"]}: {t["subject"]}')


System prompt: 8331 字符（含项目记忆）
集成示例：❶提示 + ❷工具 + ❸安全 + ❹上下文 + ❺协调


---
## 源码对照：Claude Code 的多 Agent 架构

我们实现的简化版：

```
Team Lead (主 agent)
  ├── create_task    → 创建子任务
  ├── list_tasks     → 查看进度
  ├── update_task    → 更新状态
  ├── dispatch_agent → 生成子 agent（独立 context）
  └── 基础工具       → 自己也能做事
```

Claude Code 的完整版更复杂：

```
Team Lead
  ├── TeamCreateTool   → 创建团队
  ├── TaskCreateTool   → 创建任务（支持依赖关系）
  ├── TaskUpdateTool   → 更新（支持 blocks/blockedBy）
  ├── AgentTool        → 子 agent（可选 worktree 隔离）
  ├── SendMessageTool  → agent 间消息通信
  └── 50+ 其他工具

Sub-Agent
  ├── 通过 SendMessage 向 Team Lead 汇报
  ├── 可以运行在独立 git worktree 中
  ├── 自动 idle 通知（不需要轮询）
  └── Team Lead 可以 shutdown 子 agent
```

### 关键设计差异

| 我们的版本 | Claude Code |
|-----------|-------------|
| 同步调用子 agent | 支持后台运行 |
| 子 agent 直接返回结果 | 通过 SendMessage 通信 |
| 无隔离 | Git worktree 文件隔离 |
| 简单的任务列表 | 支持依赖关系（blocks/blockedBy）|
| 固定轮数限制 | 动态资源管理 |

---
## 小结

### 本 Lab 你学到了什么

1. **Team Lead 协调模式** — 主 agent 负责拆分任务、派遣子 agent、汇总结果
2. **AgentTool（子 agent 生成）** — 子 agent 拥有独立 context，防止 token 膨胀
3. **任务管理系统** — create_task / list_tasks / update_task 跟踪工作进度
4. **防递归设计** — 子 agent 不能再生成子 agent

### 还差什么？

到目前为止我们已经有了五层 Harness，但还缺最后一块拼图：
**关掉程序再打开，一切状态都丢失了**——对话历史、任务进度、做到哪里全都没了。

→ **下一个 Lab：Lab 6 状态与持久层——补上最后一层，完成完整的 Mini Harness。**
